# Notebook 09 — Iperparametri degli LLM

**Modulo:** 09 — Costruire con le API
**Tempo stimato:** 2-3 ore

## Cosa costruirai

- Un esperimento sistematico sull'effetto di `temperature`, a parità di prompt
- Lo stesso esperimento con `top_p`, per capire come si differenzia da `temperature`
- Un confronto tra `frequency_penalty` alto e basso
- Una funzione che esplora una piccola griglia di iperparametri e ne stampa un confronto, tracciando anche il costo
- Un esercizio guidato su quando NON mescolare `temperature` e `top_p`

## Prerequisiti

```bash
pip install openai anthropic python-dotenv
```

Serve un file `.env` nella cartella del progetto con:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

> Se non hai una delle due chiavi, puoi comunque seguire il notebook: le celle
> che usano OpenAI sono quelle degli esperimenti su `frequency_penalty`
> (non disponibile nativamente su Anthropic), tutto il resto funziona con
> uno solo dei due provider.

---
## Cella 0 — Setup: installazione e caricamento delle chiavi

Esegui sempre questa cella per prima. Verifica che le chiavi vengano caricate
correttamente dal file `.env`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    """Mostra solo i primi caratteri della chiave, mai la chiave intera."""
    if chiave:
        print(f"OK  {nome}: {chiave[:8]}...")
    else:
        print(f"MANCA  {nome}: non trovata. Controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)
verifica_chiave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)

---
## Cella 1 — Wrapper multi-provider e stima del costo

Riusiamo lo stesso pattern introdotto nel Modulo 08: una funzione `chiama_llm`
che astrae le differenze tra provider, e una funzione `calcola_costo` per
stimare la spesa di ogni chiamata. Qui li ridefiniamo (ogni notebook del corso
è indipendente) ma con la stessa firma e lo stesso spirito.

In [ ]:
from openai import OpenAI
import anthropic
from typing import Literal

cliente_openai = OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None
cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None

# Modelli di riferimento 2026 (vedi STYLE.md del corso).
# Verifica sempre l'ID modello aggiornato nella documentazione ufficiale del provider.
MODELLI_DEFAULT = {
    "openai": "gpt-5-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}

# Prezzi indicativi in USD per milione di token (esempio illustrativo:
# verifica sempre i prezzi aggiornati sul sito del provider prima di
# presupuestare un progetto reale).
PREZZI = {
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
}

def calcola_costo(tokens_input: int, tokens_output: int, modello: str) -> dict:
    """Calcola il costo in USD di una chiamata, dato il modello usato."""
    if modello not in PREZZI:
        raise ValueError(f"Modello '{modello}' non presente nella tabella prezzi.")
    prezzi = PREZZI[modello]
    costo_input = (tokens_input / 1_000_000) * prezzi["input"]
    costo_output = (tokens_output / 1_000_000) * prezzi["output"]
    return {
        "modello": modello,
        "tokens_input": tokens_input,
        "tokens_output": tokens_output,
        "costo_totale_usd": round(costo_input + costo_output, 6),
    }


def chiama_llm(
    prompt: str,
    provider: Literal["openai", "anthropic"] = "openai",
    modello: str | None = None,
    temperature: float = 1.0,
    top_p: float = 1.0,
    frequency_penalty: float = 0.0,
    max_tokens: int = 200,
) -> dict:
    """Wrapper unificato per chiamare OpenAI o Anthropic con gli stessi
    iperparametri esposti in modo uniforme (frequency_penalty è ignorato
    da Anthropic, che non lo espone nella Messages API).
    """
    modello = modello or MODELLI_DEFAULT[provider]

    if provider == "openai":
        if cliente_openai is None:
            raise RuntimeError("OPENAI_API_KEY non configurata.")
        risposta = cliente_openai.chat.completions.create(
            model=modello,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            frequency_penalty=frequency_penalty,
            messages=[{"role": "user", "content": prompt}],
        )
        return {
            "testo": risposta.choices[0].message.content,
            "tokens_input": risposta.usage.prompt_tokens,
            "tokens_output": risposta.usage.completion_tokens,
            "modello": modello,
            "provider": provider,
        }

    elif provider == "anthropic":
        if cliente_anthropic is None:
            raise RuntimeError("ANTHROPIC_API_KEY non configurata.")
        risposta = cliente_anthropic.messages.create(
            model=modello,
            max_tokens=max_tokens,
            temperature=min(temperature, 1.0),  # Anthropic accetta 0-1
            top_p=top_p,
            messages=[{"role": "user", "content": prompt}],
        )
        return {
            "testo": risposta.content[0].text,
            "tokens_input": risposta.usage.input_tokens,
            "tokens_output": risposta.usage.output_tokens,
            "modello": modello,
            "provider": provider,
        }

    else:
        raise ValueError(f"Provider '{provider}' non supportato.")


print("chiama_llm() e calcola_costo() pronte.")

---
## Cella 2 — Esperimento 1: l'effetto di `temperature`

Usiamo sempre lo **stesso prompt** e variamo solo `temperature`, per isolare
l'effetto di questo unico iperparametro (metodo dello "un parametro alla
volta").

**Prima di eseguire:** cosa ti aspetti che succeda con `temperature=0.0`?
E con `temperature=1.5`? Scrivi una tua previsione prima di lanciare la cella.

In [ ]:
PROMPT_CREATIVO = "Inventa un nome per una startup che usa l'IA per aiutare gli agricoltori a irrigare in modo efficiente."
VALORI_TEMPERATURE = [0.0, 0.3, 0.7, 1.0, 1.5]

print(f"Prompt: {PROMPT_CREATIVO}\n")
print("=" * 70)

for temp in VALORI_TEMPERATURE:
    risultato = chiama_llm(PROMPT_CREATIVO, provider="openai", temperature=temp, max_tokens=40)
    print(f"temperature={temp:.1f} -> {risultato['testo'].strip()}")

print("=" * 70)
print("\nRipeti la stessa cella una seconda volta: cosa cambia con temperature=0.0")
print("rispetto a temperature=1.5? Confrontalo con la tua previsione.")

**Le tue osservazioni** (modifica questa cella):

- Con `temperature=0.0`: ...
- Con `temperature=1.5`: ...
- Rieseguendo la cella, `temperature=0.0` ha dato lo stesso risultato? E `temperature=1.5`?
- Per questo caso d'uso (naming creativo) userei temperature=... perché ...

---
## Cella 3 — Esperimento 2: l'effetto di `top_p`

Stesso principio: stesso prompt, solo `top_p` varia. Questa volta teniamo
`temperature` fissa a un valore neutro (1.0) per isolare l'effetto di `top_p`
da solo, come raccomandato nella dispensa.

In [ ]:
VALORI_TOP_P = [0.1, 0.3, 0.5, 0.8, 1.0]

print(f"Prompt: {PROMPT_CREATIVO}\n")
print("=" * 70)

for tp in VALORI_TOP_P:
    risultato = chiama_llm(PROMPT_CREATIVO, provider="openai", temperature=1.0, top_p=tp, max_tokens=40)
    print(f"top_p={tp:.1f} -> {risultato['testo'].strip()}")

print("=" * 70)

**Le tue osservazioni** (modifica questa cella):

- Con `top_p=0.1`: ...
- Con `top_p=1.0`: ...
- Confrontando questo esperimento con quello di `temperature`: la variazione
  ti sembra simile o diversa nel modo in cui cambia l'output? ...

---
## Cella 4 — Esperimento 3: `frequency_penalty` e ripetizione

Per questo esperimento usiamo un prompt che tende naturalmente a produrre
ripetizioni, per osservare meglio l'effetto della penalità. `frequency_penalty`
è disponibile solo su OpenAI: se non hai una chiave OpenAI, leggi comunque
l'output atteso descritto sotto la cella.

In [ ]:
PROMPT_RIPETITIVO = "Descrivi in un breve paragrafo i vantaggi dell'irrigazione intelligente per un'azienda agricola, insistendo sui benefici."

for fp in [0.0, 1.0, 2.0]:
    risultato = chiama_llm(
        PROMPT_RIPETITIVO,
        provider="openai",
        temperature=0.7,
        frequency_penalty=fp,
        max_tokens=90,
    )
    print(f"frequency_penalty={fp:.1f}")
    print(f"  {risultato['testo'].strip()}")
    print()

**Le tue osservazioni** (modifica questa cella):

- Con `frequency_penalty=0.0`: il testo ripete parole/concetti? ...
- Con `frequency_penalty=2.0`: il testo suona ancora naturale, o forzato? ...
- Che valore useresti in produzione per un caso simile, e perché?

---
## Cella 5 — TODO: non mescolare `temperature` e `top_p` senza capire l'effetto

La dispensa (sezione 6) avverte che combinare `temperature` alta e `top_p`
molto basso produce risultati difficili da prevedere, perché i due filtri
agiscono sulla stessa distribuzione di probabilità in modi diversi.

**Completa il TODO:** scrivi una funzione che confronta, per lo stesso prompt,
tre configurazioni:
1. Solo `temperature` alta (`top_p` di default a 1.0)
2. Solo `top_p` basso (`temperature` di default a 1.0)
3. Entrambi i valori estremi insieme (`temperature` alta E `top_p` basso)

e stampa i tre output per un confronto visivo diretto.

In [ ]:
def confronta_combinazioni(prompt: str) -> None:
    """Confronta temperature alta da sola, top_p basso da solo,
    e i due valori estremi combinati insieme."""

    # TODO: completa le tre chiamate seguenti.
    # 1. Solo temperature alta (es. 1.5), top_p di default (1.0)
    solo_temperature = chiama_llm(prompt, provider="openai", temperature=1.5, top_p=1.0, max_tokens=40)

    # 2. Solo top_p basso (es. 0.2), temperature di default (1.0)
    solo_top_p = None  # <-- sostituisci None con la chiamata a chiama_llm(...)

    # 3. Entrambi combinati: temperature=1.5 E top_p=0.2 insieme
    combinati = None  # <-- sostituisci None con la chiamata a chiama_llm(...)

    print(f"Prompt: {prompt}\n")
    print(f"Solo temperature=1.5:        {solo_temperature['testo'].strip()}")
    if solo_top_p:
        print(f"Solo top_p=0.2:               {solo_top_p['testo'].strip()}")
    if combinati:
        print(f"temperature=1.5 + top_p=0.2:  {combinati['testo'].strip()}")


confronta_combinazioni("Suggerisci un titolo per un articolo sull'IA in agricoltura.")

---
## Cella 6 — Sfida finale: mini-griglia di iperparametri con costo

Costruisci una funzione che esplora una piccola griglia di combinazioni di
`temperature` (e opzionalmente `top_p`), esegue una chiamata per ciascuna,
stampa un confronto ordinato e calcola il costo totale della griglia usando
`calcola_costo`.

In [ ]:
def esplora_griglia_iperparametri(
    prompt: str,
    valori_temperature: list[float],
    provider: str = "openai",
    max_tokens: int = 40,
) -> None:
    """Esegue una chiamata per ogni valore di temperature nella griglia,
    stampa un confronto e il costo totale accumulato."""
    costo_totale_usd = 0.0

    print(f"Prompt: {prompt}")
    print("=" * 70)

    for temp in valori_temperature:
        risultato = chiama_llm(prompt, provider=provider, temperature=temp, max_tokens=max_tokens)
        costo = calcola_costo(risultato["tokens_input"], risultato["tokens_output"], risultato["modello"])
        costo_totale_usd += costo["costo_totale_usd"]

        print(f"[temperature={temp:.1f}] {risultato['testo'].strip()}")
        print(f"   tokens: {risultato['tokens_input']} in / {risultato['tokens_output']} out"
              f" | costo: ${costo['costo_totale_usd']:.6f}")

    print("=" * 70)
    print(f"Costo totale della griglia: ${costo_totale_usd:.6f}")


esplora_griglia_iperparametri(
    prompt="Dai un consiglio pratico per ridurre lo spreco d'acqua in un vigneto.",
    valori_temperature=[0.0, 0.5, 1.0, 1.5],
)

### Estensione opzionale (se hai tempo)

Modifica `esplora_griglia_iperparametri` per accettare anche una lista di
valori `top_p` e produrre una griglia 2D (ogni combinazione temperature × top_p),
stampando i risultati in una tabella. Attenzione: il numero di chiamate cresce
rapidamente (righe × colonne) — tienilo sotto controllo con `calcola_costo`
prima di lanciare griglie grandi.

In [ ]:
# Spazio per l'estensione opzionale


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 09 — Modulo 09 del Corso Completo di IA Generativa*